# Module 8 — Capstone: The Complete OSPF Troubleshooting Agent

**Course:** Build Your First AI Agent from Scratch  
**Community:** [Autonomous MSP](https://skool.com/autonomous-msp-2162)

**What this module does:** Wires every piece you built in Modules 1–7 into one running agent.

The incident: *OSPF neighbor 3.3.3.3 on client-rtr-01 stuck in INIT state for 20 minutes. 50 branch users cannot reach the data centre.*

The agent's execution path:
1. OBSERVE — query KB, load topology, check MCP
2. RUN SHOW COMMANDS — ssh, collect output
3. REASON — identify root cause + confidence
4. APPROVAL CHECKPOINT — stop, show engineer, wait
5. APPLY — push config change
6. VERIFY — wait 30s, recheck neighbor state

## Step 1 — Install Dependencies

In [ ]:
!pip install langgraph langchain-anthropic chromadb -q

## Step 2 — Set Your API Key

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("API key loaded from Colab Secrets.")
except Exception:
    os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."  # replace with your key
    print("API key set directly.")

## Step 3 — Agent State

The capstone agent has a more specific state than the generic AgentState in Module 5. It captures the exact fields needed for OSPF incident handling.

In [ ]:
from typing import TypedDict, List, Optional, Dict


class OSPFAgentState(TypedDict):
    # ── Incident input ─────────────────────────────────────────
    device:           str    # e.g. "client-rtr-01"
    neighbor_ip:      str    # e.g. "3.3.3.3"
    neighbor_state:   str    # e.g. "INIT"
    duration_minutes: int    # how long it's been stuck

    # ── Working memory (populated during run) ──────────────────
    show_output:      Dict   # keyed by command string
    kb_hits:          List   # memory search results
    topology:         Dict   # graph query result
    recent_changes:   List   # changes in last 7 days

    # ── Reasoning ──────────────────────────────────────────────
    diagnosis:        Optional[str]   # root cause
    confidence:       float           # 0.0 to 1.0
    proposed_fix:     Optional[str]   # exact config string

    # ── Control flow ───────────────────────────────────────────
    approved:   bool   # human said yes
    applied:    bool   # config was pushed
    verified:   bool   # neighbor reached FULL after fix
    iterations: int    # loop counter
    error:      Optional[str]


print("OSPFAgentState defined with", len(OSPFAgentState.__annotations__), "fields")

## Step 4 — Demo Tools and KB Setup

All in DEMO_MODE — no real devices or Neo4j needed. The pattern is identical to what you would use with real infrastructure.

In [ ]:
import json
import time
import chromadb
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(model="claude-sonnet-4-6", temperature=0)
# temperature=0 is important for production — deterministic reasoning on live networks

# ── ChromaDB (in-memory) ──────────────────────────────────────
chroma = chromadb.Client()
kb = chroma.get_or_create_collection(
    name="ospf_incidents",
    metadata={"hnsw:space": "cosine"}
)
kb.upsert(
    ids=["INC-1955"],
    documents=["OSPF adjacency not forming after VLAN change. Area mismatch — interface moved to area 1, neighbor still in area 0."],
    metadatas=[{"resolution": "Corrected area assignment, adjacency formed in 40s", "protocol": "ospf"}]
)

# ── Demo show command outputs ─────────────────────────────────
DEMO_SHOW = {
    ("client-rtr-01", "show ip ospf neighbor detail"): json.dumps({
        "neighbor": "3.3.3.3",
        "state":    "INIT",
        "interface": "GigabitEthernet0/1",
        "dead_time": "00:00:34",
    }),
    ("client-rtr-01", "show ip ospf interface GigabitEthernet0/1"): json.dumps({
        "interface": "GigabitEthernet0/1",
        "area":      "0.0.0.0",   # Area 0 — but neighbor expects Area 1
        "hello":     10,
        "dead":      40,
        "mtu":       1500,
    }),
    ("client-rtr-01", "show ip ospf"): json.dumps({
        "router_id":  "1.1.1.1",
        "process_id": 1,
    }),
}

# Demo topology
DEMO_TOPOLOGY = {
    "client-rtr-01": {
        "role":      "branch-router",
        "neighbors": [{"device": "core-rtr-01", "interface": "Gi0/1", "expected_area": 1}],
        "services":  ["branch-office-connectivity"],
    }
}


def get_ospf_neighbors(device: str, command: str) -> dict:
    """Demo SSH tool — returns hardcoded show output."""
    return json.loads(DEMO_SHOW.get((device, command), json.dumps({"error": f"No demo data for {command}"})))


def push_config_change(device: str, config: str) -> bool:
    """Demo config push — just prints, doesn't SSH."""
    print(f"  [demo] Would push to {device}:\n{config}")
    return True


def search_kb(query: str) -> list:
    """Search ChromaDB for relevant past incidents."""
    results = kb.query(query_texts=[query], n_results=2)
    hits = []
    for i, doc in enumerate(results["documents"][0]):
        hits.append({"document": doc, "metadata": results["metadatas"][0][i]})
    return hits


def get_device_topology(device: str) -> dict:
    return DEMO_TOPOLOGY.get(device, {})


def get_recent_changes(device: str, days: int = 7) -> list:
    if device == "client-rtr-01":
        return [{"date": "4 days ago", "change": "Interface Gi0/1 area config modified"}]
    return []


print("All demo tools and KB ready.")

## Step 5 — The Prompts

Prompts are kept separate from node logic. All three prompts return structured JSON — no regex parsing, no guessing about LLM output format.

In [ ]:
OBSERVE_PROMPT = """
You are an OSPF troubleshooting agent for a managed service provider.

Current incident:
- Device: {device}
- Neighbor IP: {neighbor_ip}
- Neighbor state: {neighbor_state}
- Duration: {duration_minutes} minutes

Knowledge base results:
{kb_hits}

Topology context:
{topology}

Recent changes on this device (last 7 days):
{recent_changes}

Based on this context, list the OSPF show commands you need to diagnose this incident.
Return a JSON object with key "commands" containing a list of IOS command strings.
Return ONLY the JSON object. No explanation.
"""

REASON_PROMPT = """
You are an OSPF troubleshooting agent. You have collected the following data.

Incident: OSPF neighbor {neighbor_ip} on {device} stuck in {neighbor_state} for {duration_minutes} minutes.

Show command output:
{show_output}

Topology expectation:
{topology}

Known issues from knowledge base:
{kb_hits}

Diagnose the root cause. Return a JSON object with keys:
  root_cause  (string — one sentence)
  evidence    (list of strings — reference specific show output)
  config_fix  (string — exact IOS config lines to fix the issue)
  confidence  (float 0.0 to 1.0)

Return ONLY the JSON object. No explanation.
"""

VERIFY_PROMPT = """
You are verifying whether an OSPF fix was successful.

Original incident: neighbor {neighbor_ip} on {device} was in {neighbor_state} state.
Applied fix: {proposed_fix}

Current neighbor output after fix:
{post_fix_output}

Return a JSON object with keys:
  resolved      (bool)
  current_state (string)
  notes         (string)

Return ONLY the JSON object.
"""

print("Prompts defined.")

## Step 6 — The Four Nodes

Each node does exactly one thing. They compose into the full agent when wired together in the graph.

In [ ]:
CONFIDENCE_THRESHOLD = 0.80
MAX_ITERATIONS       = 3


def observe(state: OSPFAgentState) -> dict:
    """
    Step 1–4 of the execution path:
    - Query KB for similar past incidents
    - Load topology
    - Check recent changes
    - Ask LLM which show commands to run, then run them
    """
    print(f"\n[OBSERVE] Querying KB for: ospf {state['neighbor_ip']} {state['device']}")

    kb_hits      = search_kb(f"ospf neighbor {state['neighbor_ip']} {state['device']} INIT")
    topology     = get_device_topology(state["device"])
    changes      = get_recent_changes(state["device"], days=7)

    print(f"[OBSERVE] KB hits: {len(kb_hits)} result(s)")
    print(f"[OBSERVE] Recent changes: {len(changes)} change(s)")

    # Ask the LLM which commands to run based on the available context
    prompt   = OBSERVE_PROMPT.format(
        device=state["device"],
        neighbor_ip=state["neighbor_ip"],
        neighbor_state=state["neighbor_state"],
        duration_minutes=state["duration_minutes"],
        kb_hits=json.dumps(kb_hits, indent=2),
        topology=json.dumps(topology, indent=2),
        recent_changes=json.dumps(changes, indent=2),
    )

    response = llm.invoke(prompt)

    try:
        commands = json.loads(response.content)["commands"]
    except Exception:
        # Fallback if LLM doesn't return clean JSON
        commands = [
            "show ip ospf neighbor detail",
            "show ip ospf interface GigabitEthernet0/1",
            "show ip ospf",
        ]

    # Run each show command and collect output
    show_output = {}
    for cmd in commands:
        print(f"[OBSERVE] Running: {cmd}")
        show_output[cmd] = get_ospf_neighbors(state["device"], cmd)

    return {
        "kb_hits":        kb_hits,
        "topology":       topology,
        "recent_changes": changes,
        "show_output":    show_output,
        "iterations":     state["iterations"] + 1,
    }


def reason(state: OSPFAgentState) -> dict:
    """
    Step 6: LLM reasons over all collected data.
    Produces: diagnosis, proposed config fix, confidence score.
    Also stores the diagnosis back to the KB so future runs benefit.
    """
    print("\n[REASON] Analysing show output + topology + KB hits...")

    prompt = REASON_PROMPT.format(
        device=state["device"],
        neighbor_ip=state["neighbor_ip"],
        neighbor_state=state["neighbor_state"],
        duration_minutes=state["duration_minutes"],
        show_output=json.dumps(state["show_output"], indent=2),
        topology=json.dumps(state["topology"], indent=2),
        kb_hits=json.dumps(state["kb_hits"], indent=2),
    )

    response = llm.invoke(prompt)

    try:
        result = json.loads(response.content)
    except json.JSONDecodeError:
        # Try to extract JSON if LLM added extra text
        import re
        match = re.search(r'\{.*\}', response.content, re.DOTALL)
        result = json.loads(match.group()) if match else {"root_cause": "parse error", "config_fix": "", "confidence": 0.0}

    confidence = float(result.get("confidence", 0.0))
    diagnosis  = result.get("root_cause", "Unknown")

    print(f"[REASON] Diagnosis: {diagnosis}")
    print(f"[REASON] Confidence: {confidence:.0%}")

    # Store this diagnosis back to the KB — next incident with similar symptoms
    # will find this entry and benefit from it
    kb.upsert(
        ids=[f"{state['device']}-{state['neighbor_ip']}-{int(time.time())}"],
        documents=[f"Incident: {state['device']} neighbor {state['neighbor_ip']} "
                   f"INIT state. Root cause: {diagnosis}"],
        metadatas=[{"device": state["device"], "type": "incident_diagnosis"}]
    )

    return {
        "diagnosis":    diagnosis,
        "proposed_fix": result.get("config_fix", ""),
        "confidence":   confidence,
    }


def act(state: OSPFAgentState) -> dict:
    """
    Step 8: Human approval checkpoint.
    Show the engineer the diagnosis and proposed fix. Wait for approval.
    On approval: push the config change.
    """
    print("\n" + "-" * 50)
    print("PROPOSED FIX")
    print("-" * 50)
    print(f"Device:     {state['device']}")
    print(f"Diagnosis:  {state['diagnosis']}")
    print(f"Confidence: {state['confidence']:.0%}")
    print(f"\nConfig change:\n{state['proposed_fix']}")
    print("-" * 50)

    approval = input("\nApprove this change? (yes/no): ").strip().lower()

    if approval != "yes":
        print("Change rejected. Escalating to engineer queue.")
        return {"approved": False, "applied": False}

    push_config_change(state["device"], state["proposed_fix"])
    print(f"\nChange applied to {state['device']}.")

    return {"approved": True, "applied": True}


def verify(state: OSPFAgentState) -> dict:
    """
    Step 10: Wait for OSPF convergence, re-check neighbor state.
    In demo mode: 3 second wait instead of 30.
    """
    wait_seconds = 3  # use 30 with real devices
    print(f"\n[VERIFY] Waiting {wait_seconds} seconds for OSPF convergence...")
    time.sleep(wait_seconds)

    # Re-run the neighbor check
    post_fix = get_ospf_neighbors(state["device"], "show ip ospf neighbor detail")

    # Override demo data to show the fix worked
    post_fix["state"] = "FULL/DR"  # demo: pretend fix worked

    prompt = VERIFY_PROMPT.format(
        device=state["device"],
        neighbor_ip=state["neighbor_ip"],
        neighbor_state=state["neighbor_state"],
        proposed_fix=state["proposed_fix"],
        post_fix_output=json.dumps(post_fix, indent=2),
    )

    response = llm.invoke(prompt)

    try:
        result = json.loads(response.content)
    except Exception:
        result = {"resolved": True, "current_state": "FULL/DR", "notes": ""}

    if result.get("resolved"):
        print(f"[VERIFY] Resolved. Neighbor {state['neighbor_ip']} is now {result.get('current_state')}.")
    else:
        print(f"[VERIFY] NOT resolved. Current state: {result.get('current_state')}. Escalating.")
        if result.get("notes"):
            print(f"[VERIFY] Notes: {result['notes']}")

    return {"verified": result.get("resolved", False)}


print("All four nodes defined.")

## Step 7 — Routing Functions and Graph Assembly

Two routing functions:
- `route_after_reason` — if confidence is high enough, go to `act`; if not and iterations remain, loop back to `observe`
- `route_after_act` — if approved and applied, go to `verify`; if rejected, stop

In [ ]:
from langgraph.graph import StateGraph, END


def route_after_reason(state: OSPFAgentState) -> str:
    if state["confidence"] >= CONFIDENCE_THRESHOLD:
        print(f"  [route] confidence {state['confidence']:.0%} ≥ threshold → act")
        return "act"
    if state["iterations"] >= MAX_ITERATIONS:
        print(f"  [route] max iterations reached → escalate")
        return END
    print(f"  [route] confidence {state['confidence']:.0%} < threshold → gather more data")
    return "observe"


def route_after_act(state: OSPFAgentState) -> str:
    if state["approved"] and state["applied"]:
        return "verify"
    return END  # rejected — stop, ticket stays open for manual handling


def build_graph():
    graph = StateGraph(OSPFAgentState)

    # Add all nodes
    graph.add_node("observe", observe)
    graph.add_node("reason",  reason)
    graph.add_node("act",     act)
    graph.add_node("verify",  verify)

    # Entry point
    graph.set_entry_point("observe")

    # Fixed edges
    graph.add_edge("observe", "reason")
    graph.add_edge("verify",  END)

    # Conditional edges
    graph.add_conditional_edges(
        "reason",
        route_after_reason,
        {"act": "act", "observe": "observe", END: END},
    )
    graph.add_conditional_edges(
        "act",
        route_after_act,
        {"verify": "verify", END: END},
    )

    return graph.compile()


print("Graph compiled. Ready to run.")

## Step 8 — Run the Complete Agent

This is the capstone run. The agent will:
1. Query the KB and topology
2. Run show commands
3. Reason and identify the fix
4. **Stop and ask you to approve** (type `yes` to continue)
5. Push the fix (demo mode — just prints)
6. Wait and verify

In [ ]:
def run(device: str, neighbor_ip: str, neighbor_state: str, duration_minutes: int):
    agent = build_graph()
    start = time.time()

    initial_state: OSPFAgentState = {
        "device":           device,
        "neighbor_ip":      neighbor_ip,
        "neighbor_state":   neighbor_state,
        "duration_minutes": duration_minutes,
        "show_output":      {},
        "kb_hits":          [],
        "topology":         {},
        "recent_changes":   [],
        "diagnosis":        None,
        "confidence":       0.0,
        "proposed_fix":     None,
        "approved":         False,
        "applied":          False,
        "verified":         False,
        "iterations":       0,
        "error":            None,
    }

    result = agent.invoke(initial_state)
    elapsed = time.time() - start

    print("\n" + "=" * 50)
    print("RUN SUMMARY")
    print("=" * 50)
    print(f"Iterations:    {result.get('iterations')}")
    print(f"Confidence:    {result.get('confidence', 0):.0%}")
    print(f"Approved:      {result.get('approved')}")
    print(f"Applied:       {result.get('applied')}")
    print(f"Verified:      {result.get('verified')}")
    print(f"Wall time:     {elapsed:.0f}s")
    print(f"Diagnosis:     {result.get('diagnosis')}")
    print("=" * 50)

    return result


# The capstone scenario
result = run(
    device="client-rtr-01",
    neighbor_ip="3.3.3.3",
    neighbor_state="INIT",
    duration_minutes=20,
)

## Extension: Add BGP Support

The architecture is protocol-agnostic. To add BGP, you only need:
1. BGP-specific show tools
2. A `BGPAgentState` TypedDict with BGP-specific fields
3. BGP-specific OBSERVE and REASON prompts

The graph structure, checkpointer, and approval gate are **identical**.

In [ ]:
print("Extension paths from here:\n")

print("1. More protocols (BGP, VXLAN, MPLS, STP):")
print("   - Add protocol-specific tools in tools/bgp_tools.py")
print("   - Add protocol-specific state TypedDict")
print("   - Add protocol-specific prompts")
print("   - Same graph structure. Same approval gate.\n")

print("2. Multi-client isolation:")
print("   - Per-client ChromaDB collection: 'kb_{client_id}'")
print("   - Per-client MCP profile (credentials, IP ranges, allowed commands)")
print("   - thread_id = f'{client_id}-{device}-{neighbor_ip}'")
print("   - One codebase. Thirty clients.\n")

print("3. Automation pipeline (PagerDuty / ConnectWise):")
print("   def on_alert_received(alert):")
print("       if alert['type'] == 'ospf_neighbor_down':")
print("           result = ospf_agent.run(**alert)")
print("           post_diagnosis_to_ticket(alert['ticket_id'], result)")
print("           # Agent stops here. Human approves in the ticket system.")

print("\n4. Stages of autonomy:")
stages = [
    ("1 — Read-only assist",     "Runs show commands, posts diagnosis",          "Reviews all"),
    ("2 — Propose and wait",     "Diagnoses + proposes fix, waits in ticket",    "Approves each change"),
    ("3 — Tier-1 autonomous",    "Handles known issue types end-to-end",         "Reviews summary"),
    ("4 — Predictive",           "Detects anomalies before impact",              "Reviews weekly reports"),
]
for stage, agent_does, engineer_does in stages:
    print(f"   Stage {stage}")
    print(f"     Agent:    {agent_does}")
    print(f"     Engineer: {engineer_does}")

## You Did It

You built a production-grade AI agent from scratch:

| Module | Component |
|--------|----------|
| 01 | Mental model: scripts vs agents, ReAct pattern |
| 02 | Pure Python ReAct loop, system prompt design |
| 03 | Tool layer: BaseTool, READ/WRITE/ADMIN safety, audit log |
| 04 | Memory: ChromaDB, vector search, RAG pipeline |
| 05 | LangGraph: state machine, 4 nodes, confidence routing |
| 06 | Context: MCP standards server, Neo4j topology graph |
| 07 | Production: tracing, metrics, RBAC, loop protection |
| 08 | Capstone: all components assembled and running |

---

**Capstone Challenge:** Run the complete agent. Post in **#agent-builds**:
1. Screenshot of the RUN SUMMARY block
2. How many iterations to reach confidence threshold
3. The confidence score

Every member who completes this has built the same architecture running in production AI systems today.